# Studio Proyek 3: Fine-Tuning Transformer Pretrained untuk NLP

- Mata kuliah: Pembelajaran Mendalam (IF25-40401), Teknik Informatika ITERA
- Kelas: Deep Learning RA
- Kelompok: 15

Pada tugas ini kami mengklasifikasikan sentimen kalimat Bahasa Indonesia ke dalam tiga kelas (negatif, netral, positif) menggunakan dataset SmSA dari IndoNLU.

| No | Nama Lengkap | NIM |
|:-:|---|---|
| 1 | Muhammad Fadhilah Akbar | 123140003 |
| 2 | Muhammad Fajri Firdaus | 123140050 |

Nama berkas PDF: `STUDIO3_Kelompok15_MuhammadFadhilahAkbar_MuhammadFajriFirdaus.pdf`

| Item | Tautan |
|---|---|
| Notebook Colab | https://colab.research.google.com/drive/1yXhOE0Jsmb3oQcxS-qduwYlbVWGSeL3s?usp=sharing |
| Repositori GitHub | https://github.com/123140003-MuhammadFadhilahAkbar/Studio-Proyek-3_NLP |

## 0. Cara Menjalankan Notebook

1. Aktifkan GPU. Di Colab: Runtime > Change runtime type > T4 GPU. Di Kaggle: Settings > Accelerator > GPU T4, lalu nyalakan Internet.
2. Lakukan uji jalan terlebih dahulu. Ubah `UJI_JALAN = True` di bagian 2, lalu Run all. Mode ini hanya memakai subset kecil, 1 seed, dan 1 epoch, dan hasilnya ditulis ke berkas `*_uji_*` supaya tidak tercampur dengan hasil resmi. Mode uji tidak diblokir oleh pemeriksaan Rancangan. Jika tidak ada error, kembalikan ke `UJI_JALAN = False`.
3. Periksa isi Rancangan di bagian 3, lalu ubah `RANCANGAN_DIKONFIRMASI = True`. Selama rancangan belum dikonfirmasi atau masih ada isian kosong, notebook tidak bisa dijalankan dalam mode `"penuh"`.
4. Jalankan semua sel secara berurutan. Matriks eksperimen berisi 3 backbone × 3 strategi × 3 seed = 27 run, masing-masing sekitar 1 sampai 3 menit di GPU T4 dengan 6.000 data latih.
5. Setiap run langsung disimpan ke CSV. Di Colab, folder hasil diletakkan di Google Drive (`MyDrive/Hasil_Studio3/hasil_studio3`), sehingga hasil tidak hilang walaupun runtime di-reset. Izinkan akses Drive saat diminta di bagian 2. Jika sesi terputus, cukup jalankan ulang semua sel; run yang sudah selesai akan dilewati. Di Kaggle, hasil disimpan di `/kaggle/working`.
6. Jika pekerjaan dibagi antar-anggota, setiap anggota mengisi `JALANKAN_BACKBONE` di bagian 2 dengan backbone bagiannya, lalu CSV hasilnya digabung di bagian 8. Konfigurasi lainnya harus sama persis, dan hal ini dicek otomatis melalui kode konfigurasi.

### Hubungan bagian notebook dengan Lembar Soal dan Spesifikasi

| Bagian notebook | Ketentuan yang dipenuhi |
|---|---|
| 1. Setup Lingkungan | Hal teknis: platform Colab/Kaggle dan pustaka wajib (`transformers`, `scikit-learn`) |
| 2. Konfigurasi | Skema 1c / C.1: `max_length`, batch size, epoch, dan minimal 3 seed yang sama dikunci; ukuran subset (Slide 10) |
| 3. Rancangan | Skema 1 / C.1: matriks backbone × strategi, hipotesis H1–H3, alasan desain, variabel terkendali, kriteria Q1–Q3 |
| 4. Data | Slide 10: latih 3.000–6.000, validasi minimal 500 yang hanya dipakai untuk memilih epoch, uji memakai seluruh 500 kalimat uji resmi dan disentuh sekali per run |
| 4.3 Baseline TF-IDF dan tebakan acak | Slide 11: pembanding yang sah |
| 5. Model | Skema 3 / C.3: head replacement, full fine-tuning, partial freezing, linear probing, dengan kode freezing yang terlihat |
| 6–7. Pelatihan dan matriks | Skema 2 dan 3 / C.2 dan C.3: 3 backbone × 3 strategi × 3 seed dengan protokol yang sama |
| 8. Gabung antar-anggota | Pendukung pembagian kerja kelompok |
| 9. Benchmark inferensi | Waktu inferensi (ms/kalimat) dibandingkan dalam satu sesi |
| 10. Rekap agregat | Rata-rata ± simpangan baku antar-seed, disimpan ke `rekap_agregat_<dataset>_<kode>.csv` |
| 11–12. Visualisasi dan hipotesis | Pengujian H1–H3 dan tabel bukti untuk Q1–Q3 |
| 13. Analisis dan kesimpulan | Slide 9: jawaban 3 Pertanyaan Riset Wajib (Q1–Q3) |
| 14. Kepatuhan dan berkas | Kriteria selesai Skema 1–3, pengecekan kewajaran skor (Slide 11), dan berkas yang dikumpulkan (Slide 14) |

## 1. Setup Lingkungan

Sesuai ketentuan teknis di Spesifikasi, notebook ini dijalankan di Google Colab atau Kaggle (GPU gratis) dengan pustaka wajib HuggingFace `transformers` dan `scikit-learn`. Dataset SmSA kami unduh langsung dari repositori resmi IndoNLU, jadi pustaka `datasets` tidak diperlukan. Pustaka tersebut hanya dibutuhkan jika memakai IMDb.

In [ ]:
# Colab/Kaggle biasanya sudah punya paket ini, baris ini hanya memastikan versinya cukup baru.
%pip install -q "transformers>=4.40" scikit-learn pandas matplotlib

In [ ]:
import os
# Harus di-set sebelum torch dipakai supaya operasi cuBLAS deterministik (hasil bisa diulang).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import csv, gc, hashlib, json, math, platform, random, time, urllib.request, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed)
import sklearn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
transformers.logging.set_verbosity_error()

# Pakai algoritma deterministik; warn_only supaya operasi yang tidak punya versi deterministik tidak error
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"          # fp16 di GPU supaya lebih cepat dan hemat memori
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"

print(f"Python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"Perangkat    : {DEVICE} ({GPU_NAME}) | AMP fp16: {USE_AMP}")
if DEVICE.type != "cuda":
    print("\nPERINGATAN: GPU tidak terdeteksi. Mode 'penuh' akan sangat lambat di CPU. Aktifkan GPU T4 dulu.")

## 2. Konfigurasi Eksperimen (Variabel Terkunci)

Sesuai Skema 1 poin c dan Spesifikasi C.1, variabel berikut dikunci dan dipakai sama di semua run. Nilai yang sama juga ditulis di sel kode di bawah.

| Variabel | Nilai | Keterangan |
|---|---|---|
| `MAX_LENGTH` | 128 token | Mencakup minimal 99% kalimat SmSA (dicek di bagian 5.1) |
| `BATCH_SIZE` | 32 | Muat di memori GPU T4 untuk model base dengan fp16 |
| `EPOCHS` | 3 | Jumlah epoch maksimum; epoch terbaik dipilih berdasarkan F1 makro validasi (Slide 10) |
| `SEEDS` | 42, 123, 2024 | Seed yang sama untuk semua kombinasi |
| `TRAIN_SUBSET` | 6.000 | Subset latih terstratifikasi (Slide 10: minimal 3.000, maksimal 6.000) dengan `DATA_SEED` = 42 |
| Validasi | Seluruh data validasi resmi setelah dibersihkan (minimal 500) | Hanya dipakai untuk memilih epoch terbaik |
| Uji | Seluruh 500 kalimat uji resmi | Dipakai sekali per run, setelah epoch terbaik dipilih |
| Optimizer | AdamW, weight decay 0,01 | Warmup linear 10%, kemudian decay linear, gradient clipping 1,0 |
| Learning rate | full 2e-5, partial 3e-5, probing 5e-3 | Berbeda per strategi dan tetap di dalam rentang spesifikasi |
| `MODE` / `UJI_JALAN` | `"penuh"` / `False` | `UJI_JALAN = True` berarti uji jalan dengan 1 seed, 1 epoch, dan subset kecil (berkas `*_uji_*`) |

Semua variabel di atas digabung menjadi satu kode konfigurasi (`KODE`) berupa hash. Jika ada satu nilai yang berubah, kodenya ikut berubah sehingga berkas hasilnya terpisah dan tidak tercampur dengan hasil dari konfigurasi lain.

In [ ]:
# Konfigurasi eksperimen
# UJI_JALAN = True dipakai untuk mencoba notebook sekali dari atas sampai bawah (cek error).
UJI_JALAN = False         # True = subset kecil, 1 seed, 1 epoch; hasil masuk ke berkas *_uji_*
RANCANGAN_DIKONFIRMASI = True    # harus True agar mode "penuh" bisa jalan (lihat bagian 3)

DATASET   = "smsa"
LABELS    = ["negative", "neutral", "positive"]
LABEL2ID  = {lab: i for i, lab in enumerate(LABELS)}
ID2LABEL  = {i: lab for lab, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

# Backbone (Soal 2). Model khusus bahasa Inggris tidak dipakai untuk SmSA.
BACKBONES = {
    "mbert": {
        "hf": "bert-base-multilingual-cased",
        "nama": "mBERT",
        "peran": "Baseline utama (BERT multibahasa, 104 bahasa)",
    },
    "distilmbert": {
        "hf": "distilbert-base-multilingual-cased",
        "nama": "DistilmBERT",
        "peran": "Versi distilasi mBERT (6 lapis), untuk melihat efisiensi",
    },
    "indobert": {
        "hf": "indobenchmark/indobert-base-p1",
        "nama": "IndoBERT",
        "peran": "Monolingual Indonesia (pretraining Indo4B)",
    },
}
PAKAI_XLMR = False        # opsional: True = tambah XLM-R base (9 run lagi, tiap run sekitar 2x lebih lama)
if PAKAI_XLMR:
    BACKBONES["xlmr"] = {
        "hf": "xlm-roberta-base",
        "nama": "XLM-R",
        "peran": "Multibahasa bergaya RoBERTa (kosakata 250 ribu, parameternya jauh lebih banyak)",
    }

# Strategi fine-tuning (Soal 3). Head replacement dipakai di semua strategi.
STRATEGIES = {
    "full":    {"nama": "Full fine-tuning",                        "lr": 2e-5},
    "partial": {"nama": "Partial freezing (emb + 50% lapis awal)",  "lr": 3e-5},
    "probing": {"nama": "Linear probing (hanya head)",              "lr": 5e-3},
}

# Variabel yang dikunci
SEEDS           = [42, 123, 2024]
DATA_SEED       = 42      # seed untuk mengambil subset data
MAX_LENGTH      = 128
BATCH_SIZE      = 32
EVAL_BATCH_SIZE = 64
EPOCHS          = 3
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.1
GRAD_CLIP       = 1.0
TRAIN_SUBSET    = 6000    # Slide 10: 3.000-6.000 data latih
VALID_SUBSET    = None    # None = semua data validasi (dipakai untuk memilih epoch)
TEST_SUBSET     = None    # None = semua 500 kalimat uji resmi
SELEKSI_EPOCH   = "f1_macro_validasi"   # epoch terbaik dipilih dari F1 makro validasi

MODE = "penuh"
if UJI_JALAN:             # hanya untuk uji jalan, tidak untuk dilaporkan
    MODE = "uji"
    SEEDS, EPOCHS, TRAIN_SUBSET, VALID_SUBSET, TEST_SUBSET = SEEDS[:1], 1, 800, 200, 200

# Pembagian kerja antar-anggota (tidak mengubah kode konfigurasi)
JALANKAN_BACKBONE = None  # contoh: ["indobert"]  (None = semua backbone)
JALANKAN_STRATEGI = None  # contoh: ["full", "probing"] (None = semua strategi)

# Cek nilai konfigurasi terhadap spesifikasi
assert MODE in ("uji", "penuh")
assert MODE == "uji" or len(SEEDS) >= 3, "Spesifikasi: minimal 3 seed"
assert len(BACKBONES) >= 3, "Spesifikasi: minimal 3 backbone"
if MODE == "penuh":
    assert TRAIN_SUBSET is not None and 3000 <= TRAIN_SUBSET <= 6000, "Slide 10: data latih 3.000-6.000 sampel"
    assert TEST_SUBSET is None, "Slide 10: SmSA memakai seluruh 500 kalimat uji resmi"
assert {"mbert", "distilmbert", "indobert"} <= set(BACKBONES), "BERT, DistilBERT, IndoBERT wajib untuk SmSA"
assert "full" in STRATEGIES and ({"probing", "partial"} & set(STRATEGIES)), "Wajib: full + (probing/partial)"
for k, s in STRATEGIES.items():
    rentang = (1e-3, 5e-3) if k == "probing" else (2e-5, 5e-5)
    assert rentang[0] <= s["lr"] <= rentang[1], f"LR {k}={s['lr']} di luar rentang spesifikasi {rentang}"

# Kode konfigurasi = hash dari variabel yang dikunci
KONFIG_TERKUNCI = {
    "dataset": DATASET, "mode": MODE,
    "backbones": {k: v["hf"] for k, v in BACKBONES.items()},
    "strategies": {k: v["lr"] for k, v in STRATEGIES.items()},
    "seeds": SEEDS, "data_seed": DATA_SEED, "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO, "grad_clip": GRAD_CLIP,
    "train_subset": TRAIN_SUBSET, "valid_subset": VALID_SUBSET, "test_subset": TEST_SUBSET,
    "seleksi_epoch": SELEKSI_EPOCH, "evaluasi_akhir": "uji_resmi_smsa",
}
KODE = hashlib.md5(json.dumps(KONFIG_TERKUNCI, sort_keys=True).encode()).hexdigest()[:8]
SUFIKS = "_uji" if MODE == "uji" else ""          # supaya berkas uji jalan terpisah dari berkas resmi

In [ ]:
# Lokasi output. Di Colab langsung disimpan ke Google Drive supaya hasil tidak hilang kalau runtime putus.
# Di Kaggle disimpan di /kaggle/working.
DI_DRIVE = False
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/hasil_studio3")
else:
    OUTPUT_DIR = Path("hasil_studio3")
    try:
        from google.colab import drive
        drive.mount("/content/drive")                     # izinkan akses Drive saat diminta
        OUTPUT_DIR = Path("/content/drive/MyDrive/Hasil_Studio3/hasil_studio3")
        DI_DRIVE = True
    except ImportError:
        pass                                              # bukan Colab
    except Exception as e:
        print("PERINGATAN: gagal mount Google Drive, hasil disimpan lokal (bisa hilang bila sesi putus):", e)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = OUTPUT_DIR / "data_smsa"; DATA_DIR.mkdir(exist_ok=True)

HASIL_CSV    = OUTPUT_DIR / f"hasil_{DATASET}_{KODE}{SUFIKS}.csv"
REKAP_CSV    = OUTPUT_DIR / f"rekap_agregat_{DATASET}_{KODE}{SUFIKS}.csv"
INFER_CSV    = OUTPUT_DIR / f"inferensi_sesi_tunggal_{DATASET}_{KODE}{SUFIKS}.csv"
PRED_JSON    = OUTPUT_DIR / f"prediksi_{DATASET}_{KODE}{SUFIKS}.json"
KONFIG_JSON  = OUTPUT_DIR / f"konfigurasi_{DATASET}_{KODE}{SUFIKS}.json"
BASELINE_CSV = OUTPUT_DIR / f"baseline_{DATASET}_{KODE}{SUFIKS}.csv"
KONFIG_JSON.write_text(json.dumps(KONFIG_TERKUNCI, indent=2))

n_run = len(BACKBONES) * len(STRATEGIES) * len(SEEDS)
print(f"MODE = {MODE} | KODE konfigurasi = {KODE}{SUFIKS}")
print(f"Matriks: {len(BACKBONES)} backbone × {len(STRATEGIES)} strategi × {len(SEEDS)} seed = {n_run} run")
print(f"Output : {OUTPUT_DIR.resolve()}" + ("  (Google Drive)" if DI_DRIVE else ""))
if MODE == "uji":
    print("Catatan: UJI_JALAN aktif, hasil masuk ke berkas *_uji_* dan tidak untuk dilaporkan.")

## 3. Rancangan Eksperimen (Soal 1 / Skema 1)

Sesuai Skema 1 dan Spesifikasi C.1, rancangan ini kami tulis sebelum eksperimen pertama dijalankan. Isinya meliputi dataset dan ukuran subset, matriks backbone × strategi, variabel yang dikunci, hipotesis H1–H3, dan alasan desain. Kriteria untuk menjawab Q1–Q3 juga kami tetapkan di sini, supaya kriteria tersebut tidak dipilih setelah melihat hasil.

### 3.1 Dataset dan alasan pemilihan

Kami memakai Opsi A, yaitu SmSA dari IndoNLU (Wilie dkk., 2020), yang juga merupakan opsi yang disarankan Lembar Soal. Dataset ini berisi kalimat dan ulasan Bahasa Indonesia yang dilabeli manusia sebagai negatif, netral, atau positif. Kami memilihnya karena sesuai dengan konteks studi (sentimen teks Indonesia) dan memungkinkan IndoBERT ikut diuji.

Pembagian data mengikuti Slide 10:

| Himpunan | Sumber | Ukuran | Peran |
|---|---|:-:|---|
| Latih | `train_preprocess.tsv` resmi, dibersihkan, lalu diambil subset terstratifikasi (`DATA_SEED` = 42) | 6.000 | Melatih model (sama untuk semua run) |
| Validasi | `valid_preprocess.tsv` resmi, dibersihkan | sekitar 1.200 (minimal 500) | Hanya untuk memilih epoch terbaik (F1 makro) |
| Uji | `test_preprocess.tsv` resmi (berlabel) | 500 (seluruhnya) | Dipakai sekali per run, setelah epoch terbaik dipilih |

Beberapa hal yang kami perhatikan:
- Kebocoran data dicegah. Teks yang muncul di data uji atau validasi dibuang dari data latih, dan teks validasi yang muncul di data latih atau uji dibuang dari validasi. Data uji tidak dikurangi sama sekali.
- Sebaran kelas tidak seimbang dan berbeda antar-himpunan. Data latih didominasi kelas positif (sekitar 58%) dengan kelas netral hanya sekitar 10%, sedangkan data uji resmi berisi sekitar 41% negatif, 18% netral, dan 42% positif. Karena itu F1 makro kami jadikan metrik utama, sementara akurasi hanya sebagai pelengkap.
- Ukuran subset 6.000 adalah batas atas yang diizinkan. Dengan ukuran ini kelas netral masih memiliki sekitar 620 contoh dan hasil antar-seed lebih stabil, sementara satu run tetap selesai dalam 1 sampai 3 menit di GPU T4 (27 run sekitar 1 jam).

### 3.2 Matriks eksperimen (backbone × strategi), 3 seed per sel

| Backbone / Strategi | Full FT (lr 2e-5) | Partial freezing (lr 3e-5) | Linear probing (lr 5e-3) |
|---|:-:|:-:|:-:|
| mBERT (`bert-base-multilingual-cased`) | 3 seed | 3 seed | 3 seed |
| DistilmBERT (`distilbert-base-multilingual-cased`) | 3 seed | 3 seed | 3 seed |
| IndoBERT (`indobenchmark/indobert-base-p1`) | 3 seed | 3 seed | 3 seed |
| XLM-R (`xlm-roberta-base`), opsional | (3 seed) | (3 seed) | (3 seed) |

Totalnya 27 run, atau 36 run jika XLM-R diaktifkan. Setiap backbone dicoba dengan semua strategi, sehingga perbandingan antar-backbone maupun antar-strategi bisa dilakukan secara setara (apple-to-apple). Di luar matriks ini kami juga memakai dua pembanding, yaitu TF-IDF + regresi logistik dan tebakan acak (Slide 11).

### 3.3 Variabel terkendali (dikunci, sama di semua run)

`max_length` = 128, batch size = 32, epoch maksimum = 3 (epoch terbaik dipilih lewat validasi), seed = 42, 123, dan 2024, data latih = 6.000 (subset yang sama), AdamW (weight decay 0,01), warmup 10%, gradient clipping 1,0, dan fp16. Learning rate hanya dibedakan per strategi, tidak per backbone.

### 3.4 Hipotesis

- H1 (bahasa pretraining): pada full fine-tuning, IndoBERT menghasilkan F1 makro lebih tinggi daripada mBERT karena IndoBERT dilatih khusus pada teks Bahasa Indonesia.
- H2 (efisiensi distilasi): DistilmBERT memiliki waktu latih dan waktu inferensi lebih rendah daripada mBERT, dengan penurunan F1 makro paling banyak 3 poin. Besarnya penghematan diukur, bukan diasumsikan.
- H3 (strategi): urutan performa adalah full ≥ partial > probing. Partial mendekati full (selisih paling banyak 2 poin) walaupun parameter yang dilatih jauh lebih sedikit, sedangkan probing turun paling jauh.

### 3.5 Alasan desain

- Head replacement: head MLM/NSP bawaan dibuang dan diganti head klasifikasi baru dengan 3 keluaran sesuai jumlah label SmSA.
- Full fine-tuning menjadi baseline strategi. Partial freezing dan linear probing dipakai untuk melihat seberapa banyak lapis yang benar-benar perlu dilatih ulang, yang penting jika anggaran komputasi terbatas.
- Learning rate per strategi: full 2e-5 dan partial 3e-5 (rentang 2e-5 sampai 5e-5) agar pengetahuan pretrained tidak rusak. Probing memakai 5e-3 (batas atas rentang 1e-3 sampai 5e-3) karena hanya head acak yang dilatih dan anggarannya hanya 3 epoch (sekitar 560 langkah), sehingga perlu langkah yang besar agar tidak underfitting.
- Epoch terbaik dipilih dengan F1 makro validasi. Data uji tidak pernah dipakai untuk mengambil keputusan apa pun.
- Selain skor, kami juga mencatat waktu latih, waktu inferensi (ms/kalimat), jumlah parameter total dan yang dilatih, serta memori puncak. Data ini dipakai untuk menimbang anggaran komputasi dan kebutuhan pemakaian real-time.

### 3.6 Pertanyaan riset wajib (Slide 9) dan kriteria jawabannya

Kriteria berikut kami tetapkan sebelum eksperimen dijalankan.
- Q1: backbone mana yang memberi trade-off akurasi dan efisiensi terbaik? Jawaban memakai F1 makro ± SD, waktu latih, waktu inferensi (ms/kalimat, diukur dalam satu sesi), dan jumlah parameter. Backbone yang tidak kalah sekaligus di F1 dan latensi oleh konfigurasi lain (batas Pareto) menjadi kandidat.
- Q2: apakah strategi hemat komputasi (probing/partial) cukup kompetitif dibanding full fine-tuning? Selisih F1 makro dianggap bermakna jika |Δ| > SD₁ + SD₂ (jumlah simpangan baku antar-seed kedua kelompok); jika tidak, kedua strategi dianggap belum berbeda. Probing atau partial disebut kompetitif jika selisihnya tidak bermakna atau paling banyak 2 poin, dan skornya tetap di atas baseline TF-IDF.
- Q3: kombinasi apa yang kami rekomendasikan? Pertama, F1 makro harus di atas baseline TF-IDF. Kedua, kombinasi diberi skor gabungan 0,5 × F1 makro + 0,25 × kecepatan inferensi + 0,25 × hemat waktu latih, dengan setiap komponen dinormalisasi min–max ke rentang 0–1. Ketiga, jika skornya hampir sama, kami memilih kombinasi yang F1-nya tidak berbeda bermakna dari yang terbaik dan parameternya lebih sedikit, karena lebih mudah dijelaskan dan dipasang.

### 3.7 Urutan pengerjaan

1. Jalankan bagian 1–3, sesuaikan Rancangan, lalu ubah `RANCANGAN_DIKONFIRMASI = True`.
2. Lakukan uji jalan dengan `UJI_JALAN = True` lalu Run all (hasilnya masuk ke berkas terpisah `*_uji_*`).
3. Ubah kembali `UJI_JALAN = False`, lalu Run all untuk eksperimen resmi (27 run).
4. Baca hasil di bagian 10–12, tulis analisis dan jawaban Q1–Q3 di bagian 13, lalu jalankan bagian 14 sampai semua pemeriksaan lolos.

In [ ]:
# Ringkasan rancangan dari sel markdown di atas. Mode "penuh" tidak bisa dijalankan (bagian 7)
# kalau ada isian kosong atau RANCANGAN_DIKONFIRMASI masih False.
RANCANGAN = {
    "dataset": "SmSA (IndoNLU, Opsi A), sentimen 3 kelas Bahasa Indonesia; latih 6.000 (subset), validasi resmi untuk pilih epoch, uji resmi 500 disentuh sekali per run",
    "alasan_dataset_dan_subset": "Sesuai konteks sentimen teks Indonesia & dilabeli manusia; 6.000 = batas atas Slide 10 agar netral cukup terwakili dan 27 run tetap muat anggaran GPU T4",
    "matriks": "3 backbone (mBERT, DistilmBERT, IndoBERT) x 3 strategi (full, partial, probing) x 3 seed = 27 run",
    "H1": "IndoBERT > mBERT (F1 makro, full FT) karena pretraining berbahasa Indonesia",
    "H2": "DistilmBERT lebih cepat (latih & inferensi) dari mBERT dengan penurunan F1 makro <= 3 poin; besaran penghematan diuji, bukan diasumsikan",
    "H3": "full >= partial > probing; partial mendekati full (<= 2 poin) dengan parameter latih jauh lebih sedikit",
    "alasan_desain": "Head diganti ke 3 kelas; full sebagai baseline; partial & probing menguji kebutuhan adaptasi lapis; LR per strategi",
    "variabel_terkendali": f"max_length={MAX_LENGTH}, batch={BATCH_SIZE}, epoch maks={EPOCHS} (pilih via validasi), seeds={SEEDS}, latih={TRAIN_SUBSET}, AdamW wd={WEIGHT_DECAY}, warmup={WARMUP_RATIO}, clip={GRAD_CLIP}",
    "pembanding": "Baseline TF-IDF + regresi logistik (latih penuh & subset yang sama) dan tebakan acak; dievaluasi pada uji resmi yang sama",
    "kriteria_Q1_Q3": "Q1: F1±SD, waktu latih, inferensi, parameter (Pareto F1 vs latensi); Q2: bermakna bila |selisih| > SD1+SD2; Q3: syarat > TF-IDF, skor 0,5 F1 + 0,25 inferensi + 0,25 latih (min-max)",
}

def cek_rancangan(tegas=True):
    # tegas=True: hentikan notebook kalau belum lengkap; tegas=False: cukup tampilkan peringatan
    kosong = [k for k, v in RANCANGAN.items() if not str(v).strip() or "TODO" in str(v).upper()]
    if kosong:
        pesan = (f"Isian Rancangan masih kosong: {kosong}\n"
                 "   Lengkapi sel Rancangan (bagian 3) sesuai pemikiran kelompok.")
        raise RuntimeError(pesan) if tegas else print("PERINGATAN: " + pesan)
    if not RANCANGAN_DIKONFIRMASI:
        pesan = ("RANCANGAN_DIKONFIRMASI masih False.\n"
                 "   Tinjau isi RANCANGAN, sesuaikan bila perlu, lalu ubah menjadi True.")
        raise RuntimeError(pesan) if tegas else print("PERINGATAN: " + pesan)
    print("Rancangan lengkap dan sudah dikonfirmasi.")
    (OUTPUT_DIR / f"rancangan_{DATASET}_{KODE}{SUFIKS}.json").write_text(json.dumps(RANCANGAN, indent=2, ensure_ascii=False))

cek_rancangan(tegas=False)
pd.DataFrame(RANCANGAN.items(), columns=["Komponen", "Isi"]).style.set_properties(**{"text-align": "left"})

### 3.8 Pemeriksaan rancangan dan konfigurasi sebelum eksperimen

Sel berikut memeriksa beberapa hal sebelum ada satu run pun yang dijalankan: isian Rancangan sudah lengkap, `RANCANGAN_DIKONFIRMASI` sudah bernilai True, ketiga backbone wajib tersedia, tidak ada model khusus Bahasa Inggris yang dipakai untuk SmSA, dan learning rate setiap strategi berada di dalam rentang spesifikasi. Urutan jumlah parameter yang dilatih (probing < partial < full) diperiksa di bagian 5.3, masih sebelum eksperimen di bagian 7. Pemeriksaan yang membutuhkan hasil run dilakukan di bagian 14.

In [ ]:
# Pemeriksaan rancangan dan konfigurasi sebelum pelatihan dijalankan
print("Pemeriksaan rancangan dan konfigurasi (sebelum eksperimen):")
MODEL_INGGRIS = ("bert-base-uncased", "distilbert-base-uncased", "roberta-base")
kosong_R = [k for k, v in RANCANGAN.items() if not str(v).strip() or "TODO" in str(v).upper()]

cek_statis = {
    "Soal 1: isian Rancangan lengkap (tidak ada yang kosong)": not kosong_R,
    "Soal 1: Rancangan sudah dikonfirmasi kelompok": RANCANGAN_DIKONFIRMASI,
    "Soal 1: variabel terkunci tertulis lengkap (max_length, batch, epoch, seed)":
        all([MAX_LENGTH, BATCH_SIZE, EPOCHS]) and len(SEEDS) >= 3,
    "Soal 2: minimal 3 backbone tersedia": len(BACKBONES) >= 3,
    "Soal 2: BERT + DistilBERT + IndoBERT tersedia": {"mbert", "distilmbert", "indobert"} <= set(BACKBONES),
    "Soal 2: tidak ada model khusus bahasa Inggris di SmSA": not any(v["hf"] in MODEL_INGGRIS for v in BACKBONES.values()),
    "Soal 2: bahasa model cocok dengan dataset (SmSA)": all(
        ("multilingual" in v["hf"]) or ("indonesia" in v["peran"].lower()) for v in BACKBONES.values()),
    "Soal 3: full fine-tuning tersedia (wajib)": "full" in STRATEGIES,
    "Soal 3: probing dan/atau partial freezing tersedia": bool({"probing", "partial"} & set(STRATEGIES)),
    "Soal 3: semua backbone dicoba dengan semua strategi": (
        (JALANKAN_STRATEGI is None or set(JALANKAN_STRATEGI) >= set(STRATEGIES)) and len(STRATEGIES) >= 2),
    "Soal 3: head replacement dipakai di semua run": True,
    "Soal 3: LR per strategi di dalam rentang spesifikasi": all(
        (1e-3 <= s["lr"] <= 5e-3) if k == "probing" else (2e-5 <= s["lr"] <= 5e-5)
        for k, s in STRATEGIES.items()),
    "Hal teknis: >= 3 seed identik untuk semua kombinasi": (len(SEEDS) >= 3) if MODE == "penuh" else True,
    "Slide 10: data latih 3.000-6.000 sampel": (MODE == "uji") or (TRAIN_SUBSET is not None and 3000 <= TRAIN_SUBSET <= 6000),
    "Slide 10: uji = seluruh uji resmi (tanpa subset)": (MODE == "uji") or TEST_SUBSET is None,
    "Slide 10: validasi hanya untuk memilih epoch": SELEKSI_EPOCH == "f1_macro_validasi",
}
for k, v in cek_statis.items():
    print(("[OK]   " if v else "[GAGAL]"), k)
if kosong_R:
    print("   Isian Rancangan yang masih kosong:", kosong_R)
if not RANCANGAN_DIKONFIRMASI:
    print("   PERINGATAN: set RANCANGAN_DIKONFIRMASI = True sebelum MODE \"penuh\" (mode uji tidak diblokir).")
print("   Pemeriksaan urutan parameter probing < partial < full: lihat bagian 5.3.")

## 4. Data: Pembagian dan Pembersihan

Sesuai Slide 10 dan ketentuan pembagian data di Spesifikasi, SmSA dipakai dengan pembagian resmi (latih, validasi, uji). Langkah pembersihannya sebagai berikut.

1. Pada data latih dan validasi, kami membuang teks kosong atau berlabel tidak valid, teks yang sama tetapi labelnya bertentangan, dan teks duplikat.
2. Data uji resmi (500 kalimat berlabel dari `test_preprocess.tsv`) dipakai utuh tanpa membuang satu baris pun, sesuai Slide 10.
3. Untuk mencegah kebocoran, teks latih yang juga muncul di data uji atau validasi dibuang dari data latih, dan teks validasi yang muncul di data latih atau uji dibuang dari validasi.
4. Subset latih 6.000 diambil secara terstratifikasi dengan `DATA_SEED` yang tetap, sehingga subsetnya sama untuk semua run.

Sel di bawah mencetak jumlah baris yang dibuang pada setiap tahap, sehingga proses pembersihan bisa diperiksa.

In [ ]:
URL_BASE = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/"
BERKAS = {"train": "train_preprocess.tsv", "valid": "valid_preprocess.tsv",
          "test": "test_preprocess.tsv"}   # data uji resmi yang berlabel (bukan *_masked_label.tsv)

def unduh_dan_muat(split):
    path = DATA_DIR / BERKAS[split]
    if not path.exists():
        urllib.request.urlretrieve(URL_BASE + BERKAS[split], path)
    df = pd.read_csv(path, sep="\t", header=None, names=["text", "label"],
                     quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False)
    return df

raw_train = unduh_dan_muat("train")
raw_valid = unduh_dan_muat("valid")
raw_test  = unduh_dan_muat("test")
print(f"Data mentah: latih {len(raw_train)} | validasi: {len(raw_valid)} | uji: {len(raw_test)}")
print("Label uji resmi:", raw_test["label"].value_counts().to_dict())
raw_train.head()

In [ ]:
def bersihkan(df):
    lap = {"awal": len(df)}
    df = df.copy()
    df["text"] = df["text"].str.strip()
    df["label"] = df["label"].str.strip().str.lower()
    df = df[(df["text"] != "") & (df["label"].isin(LABELS))]
    lap["kosong/label_tak_valid"] = lap["awal"] - len(df)
    # teks yang sama tapi labelnya berbeda dibuang semuanya
    n_label = df.groupby("text")["label"].nunique()
    konflik = set(n_label[n_label > 1].index)
    sebelum = len(df); df = df[~df["text"].isin(konflik)]
    lap["baris_label_bertentangan"] = sebelum - len(df)
    # duplikat (labelnya pasti sama) disisakan satu
    sebelum = len(df); df = df.drop_duplicates(subset="text", keep="first")
    lap["duplikat"] = sebelum - len(df)
    return df.reset_index(drop=True), lap

train_df, lap_train = bersihkan(raw_train)
valid_df, lap_valid = bersihkan(raw_valid)

# Data uji resmi dipakai utuh (Slide 10), hanya spasinya yang dirapikan.
test_df = raw_test.copy()
test_df["text"] = test_df["text"].str.strip(); test_df["label"] = test_df["label"].str.strip().str.lower()
assert test_df["label"].isin(LABELS).all(), "Ada label uji yang tidak dikenali"
lap_test = {"awal": len(raw_test), "kosong/label_tak_valid": 0, "baris_label_bertentangan": 0, "duplikat": 0}
teks_uji = set(test_df["text"])

# Cegah kebocoran: teks uji/validasi dibuang dari latih, teks latih/uji dibuang dari validasi
bocor_tr = train_df["text"].isin(teks_uji | set(valid_df["text"]))
lap_train["bocor_ke_uji/valid"] = int(bocor_tr.sum()); train_df = train_df[~bocor_tr].reset_index(drop=True)
bocor_v = valid_df["text"].isin(set(train_df["text"]) | teks_uji)
lap_valid["bocor_ke_uji/valid"] = int(bocor_v.sum()); valid_df = valid_df[~bocor_v].reset_index(drop=True)
lap_test["bocor_ke_uji/valid"] = 0
for lap, d in [(lap_train, train_df), (lap_valid, valid_df), (lap_test, test_df)]:
    lap["akhir"] = len(d)

laporan_bersih = pd.DataFrame({"latih": lap_train, "validasi (pilih epoch)": lap_valid, "uji resmi": lap_test})
laporan_bersih

In [ ]:
# Subset terstratifikasi dengan DATA_SEED tetap, jadi sama untuk semua run
def ambil_subset(df, n):
    if n is None or n >= len(df):
        return df.reset_index(drop=True)
    sub, _ = train_test_split(df, train_size=n, stratify=df["label"], random_state=DATA_SEED)
    return sub.reset_index(drop=True)

train_full_df = train_df.copy()                 # untuk baseline TF-IDF dengan data latih penuh
train_df = ambil_subset(train_df, TRAIN_SUBSET)
valid_df = ambil_subset(valid_df, VALID_SUBSET)
test_df  = ambil_subset(test_df, TEST_SUBSET)   # TEST_SUBSET=None berarti semua 500 kalimat
y_train = train_df["label"].map(LABEL2ID).to_numpy()
y_valid = valid_df["label"].map(LABEL2ID).to_numpy()
y_test  = test_df["label"].map(LABEL2ID).to_numpy()

# Pastikan tidak ada teks yang sama di dua himpunan
OVERLAP = {
    "latih & validasi": len(set(train_df.text) & set(valid_df.text)),
    "latih & uji": len(set(train_df.text) & set(test_df.text)),
    "validasi & uji": len(set(valid_df.text) & set(test_df.text)),
}
assert sum(OVERLAP.values()) == 0, f"Kebocoran data: {OVERLAP}"
print(f"Data dipakai: latih {len(train_df)} | validasi: {len(valid_df)} | uji: {len(test_df)}")
print("Tumpang-tindih antar-himpunan:", OVERLAP)

dist = pd.DataFrame({
    "latih": train_df["label"].value_counts().reindex(LABELS),
    "validasi": valid_df["label"].value_counts().reindex(LABELS),
    "uji": test_df["label"].value_counts().reindex(LABELS),
})
dist_pct = (dist / dist.sum() * 100).round(1).add_suffix(" (%)")
display(pd.concat([dist, dist_pct], axis=1))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
dist_pct.set_axis(dist.columns, axis=1).plot.bar(ax=axes[0], rot=0, title="Distribusi kelas (%) per himpunan")
axes[0].set_ylabel("% kalimat")
train_df["text"].str.split().str.len().plot.hist(bins=50, ax=axes[1], title="Panjang kalimat latih (kata)")
axes[1].set_xlabel("jumlah kata")
plt.tight_layout(); plt.show()

### 4.3 Pembanding: baseline TF-IDF dan tebakan acak (Slide 11)

Lembar Soal menyebutkan bahwa tidak ada angka yang dianggap benar. Hasil transformer dibandingkan dengan baseline TF-IDF buatan sendiri, tebakan acak, dan simpangan antar-seed. Baseline dilatih tanpa tuning (hiperparameter tetap) dan dievaluasi pada data uji resmi yang sama dengan transformer:
- TF-IDF + regresi logistik dengan seluruh data latih, sebagai padanan acuan di Lembar Soal (F1 makro sekitar 0,73 dengan data latih penuh).
- TF-IDF + regresi logistik dengan subset 6.000 yang sama persis dengan data latih transformer. Ini pembanding yang paling adil.
- Tebakan acak (seragam) dan tebakan kelas mayoritas, dirata-ratakan atas seed yang sama.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier

def nilai_baseline(nama, y_pred, n_latih):
    return {"model": nama, "n_latih": n_latih, "accuracy": accuracy_score(y_test, y_pred),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0)}

baris_bl = []
for nama, df_l in [("TF-IDF + LogReg (latih penuh)", train_full_df), ("TF-IDF + LogReg (subset latih yang sama)", train_df)]:
    t0 = time.perf_counter()
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000, C=1.0))          # tanpa tuning
    pipe.fit(df_l["text"], df_l["label"].map(LABEL2ID))
    r = nilai_baseline(nama, pipe.predict(test_df["text"]), len(df_l)); r["train_time_s"] = time.perf_counter() - t0
    baris_bl.append(r)
for strategi_dummy, nama in [("uniform", "Tebakan acak (seragam)"), ("most_frequent", "Kelas mayoritas")]:
    skor = [nilai_baseline(nama, DummyClassifier(strategy=strategi_dummy, random_state=s)
                           .fit(train_df["text"], y_train).predict(test_df["text"]), len(train_df)) for s in SEEDS]
    baris_bl.append({"model": nama, "n_latih": len(train_df),
                     "accuracy": np.mean([s["accuracy"] for s in skor]), "f1_macro": np.mean([s["f1_macro"] for s in skor]),
                     "train_time_s": 0.0})
BASELINE_DF = pd.DataFrame(baris_bl)
BASELINE_DF.to_csv(BASELINE_CSV, index=False)
F1_TFIDF        = float(BASELINE_DF.loc[0, "f1_macro"])     # acuan utama (Slide 11)
F1_TFIDF_SUBSET = float(BASELINE_DF.loc[1, "f1_macro"])
F1_ACAK         = float(BASELINE_DF.loc[2, "f1_macro"])
print(f"Acuan pembanding pada uji resmi: TF-IDF (penuh) F1 = {F1_TFIDF:.4f} | TF-IDF (subset) = {F1_TFIDF_SUBSET:.4f} | acak = {F1_ACAK:.4f}")
BASELINE_DF.round(4)

## 5. Model: Head Replacement dan Strategi Fine-Tuning (Soal 3)

Sesuai Spesifikasi C.3, head replacement dilakukan di semua run, full fine-tuning dipakai sebagai baseline, dan dibandingkan dengan partial freezing serta linear probing.

### 5.1 Tokenisasi per backbone

Setiap backbone memakai tokenizer-nya masing-masing. Kami juga menghitung persentase kalimat yang terpotong pada `MAX_LENGTH` untuk memastikan nilai `max_length` yang dipilih sudah cukup (C.1).

In [ ]:
class DatasetTeks(Dataset):
    def __init__(self, encodings, labels):
        self.enc, self.labels = encodings, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item

TOKENIZERS, COLLATORS, DATA, STAT_TOKEN = {}, {}, {}, []
for bk, cfg in BACKBONES.items():
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    TOKENIZERS[bk] = tok
    COLLATORS[bk] = DataCollatorWithPadding(tok)     # padding per batch
    enc_tr = tok(train_df["text"].tolist(), truncation=True, max_length=MAX_LENGTH)
    enc_va = tok(valid_df["text"].tolist(), truncation=True, max_length=MAX_LENGTH)
    enc_te = tok(test_df["text"].tolist(),  truncation=True, max_length=MAX_LENGTH)
    DATA[bk] = {"train": DatasetTeks(enc_tr, y_train), "valid": DatasetTeks(enc_va, y_valid),
                "test": DatasetTeks(enc_te, y_test)}
    panjang = np.array([len(x) for x in tok(train_df["text"].tolist(), truncation=False)["input_ids"]])
    STAT_TOKEN.append({"backbone": bk, "ukuran_vocab": tok.vocab_size,
                       "rata2_token": panjang.mean().round(1), "p95_token": int(np.percentile(panjang, 95)),
                       "maks_token": int(panjang.max()),
                       f"% terpotong (>{MAX_LENGTH})": round((panjang > MAX_LENGTH).mean() * 100, 2)})

contoh = train_df["text"].iloc[0]
print("Contoh kalimat:", contoh[:120], "...\n")
for bk, tok in TOKENIZERS.items():
    print(f"{bk:12s}:", tok.tokenize(contoh)[:15])
pd.DataFrame(STAT_TOKEN)

### 5.2 Head replacement (dilakukan di semua run)

`AutoModelForSequenceClassification` memuat badan model (backbone) pretrained, membuang head pretraining (MLM/NSP), lalu memasang head klasifikasi baru berukuran `hidden_size` ke `NUM_LABELS` (3 kelas) dengan bobot acak. Opsi `output_loading_info=True` kami pakai untuk menunjukkan bahwa bobot head memang baru. Bobot tersebut muncul di `missing_keys`, yang berarti tidak ada di checkpoint pretrained.

In [ ]:
def buat_model(hf_name):
    # Muat backbone pretrained lalu pasang head klasifikasi baru (head replacement)
    model, info = AutoModelForSequenceClassification.from_pretrained(
        hf_name, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True, output_loading_info=True)
    assert model.config.num_labels == NUM_LABELS
    return model, info

def modul_head(model):
    # Head = modul di luar backbone (base_model)
    return {n: m for n, m in model.named_children() if n != model.base_model_prefix and n != "dropout"}

for bk, cfg in BACKBONES.items():
    m, info = buat_model(cfg["hf"])
    print(f"{cfg['nama']} ({cfg['hf']})")
    for nama, mod in modul_head(m).items():
        print(f"  head baru {nama}: {mod}")
    print(f"  bobot baru (tidak ada di checkpoint): {info['missing_keys']}\n")
    del m
gc.collect()

### 5.3 Strategi fine-tuning: kode pembekuan lapis (freezing)

Sesuai C.3, kami menjalankan tiga strategi agar perbandingannya setara: `full` (wajib, sebagai baseline), `partial` (partial freezing), dan `probing` (linear probing).

| Strategi | Bagian yang dibekukan | Bagian yang dilatih |
|---|---|---|
| Full fine-tuning | tidak ada | semua parameter |
| Partial freezing | embedding dan separuh lapis awal encoder (6 dari 12 lapis; pada DistilBERT 3 dari 6 lapis) | separuh lapis atas, pooler, dan head |
| Linear probing | seluruh backbone (embedding, semua lapis, pooler) | hanya head |

Pembekuan dilakukan dengan `param.requires_grad = False`, sehingga optimizer hanya menerima parameter yang `requires_grad=True`.

Catatan: pada DistilBERT, head klasifikasi terdiri dari dua modul berurutan (`pre_classifier` lalu `classifier`), sedangkan pada BERT dan IndoBERT hanya satu modul linear (`classifier`). Dua modul di sini bukan dua lapis transformer. Pada probing, pooler BERT ikut dibekukan karena termasuk bagian dari backbone pretrained.

In [ ]:
def ambil_bagian_backbone(model):
    # Ambil backbone, embedding, dan daftar lapis encoder (BERT, DistilBERT, XLM-R)
    base = getattr(model, model.base_model_prefix)               # .bert / .distilbert / .roberta
    if hasattr(base, "encoder"):                                  # BERT, IndoBERT, XLM-R
        layers = base.encoder.layer
    elif hasattr(base, "transformer"):                            # DistilBERT
        layers = base.transformer.layer
    else:
        raise ValueError(f"Arsitektur tidak dikenali: {type(base)}")
    return base, base.embeddings, layers

def terapkan_strategi(model, strategi):
    base, embeddings, layers = ambil_bagian_backbone(model)
    for p in model.parameters():                                  # awalnya semua parameter bisa dilatih
        p.requires_grad = True

    if strategi == "full":
        pass                                                      # tidak ada yang dibekukan
    elif strategi == "partial":
        for p in embeddings.parameters():                         # bekukan embedding
            p.requires_grad = False
        n_beku = len(layers) // 2                                 # 12 -> 6, DistilBERT 6 -> 3
        for layer in layers[:n_beku]:                             # bekukan separuh lapis awal
            for p in layer.parameters():
                p.requires_grad = False
    elif strategi == "probing":
        for p in base.parameters():                               # bekukan seluruh backbone
            p.requires_grad = False
    else:
        raise ValueError(strategi)

    for mod in modul_head(model).values():                        # head selalu dilatih
        for p in mod.parameters():
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    dilatih = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"params_total": total, "params_trainable": dilatih,
            "persen_trainable": round(100 * dilatih / total, 3), "n_lapis": len(layers)}

In [ ]:
# Cek: jumlah parameter yang dilatih harus probing < partial < full di setiap backbone
URUTAN = [s for s in ["probing", "partial", "full"] if s in STRATEGIES]
baris = []
for bk, cfg in BACKBONES.items():
    m, _ = buat_model(cfg["hf"])
    info = {st: terapkan_strategi(m, st) for st in URUTAN}
    dilatih = [info[st]["params_trainable"] for st in URUTAN]
    assert all(a < b for a, b in zip(dilatih, dilatih[1:])), f"Urutan parameter salah untuk {bk}: {dilatih}"
    for st in URUTAN:
        baris.append({"backbone": bk, "strategi": st, "n_lapis": info[st]["n_lapis"],
                      "params_total (juta)": round(info[st]["params_total"] / 1e6, 1),
                      "params_dilatih": info[st]["params_trainable"],
                      "% dilatih": info[st]["persen_trainable"]})
    del m; gc.collect()
print("Urutan parameter dilatih valid: " + " < ".join(URUTAN) + " untuk semua backbone")
TABEL_PARAM = pd.DataFrame(baris)
TABEL_PARAM

## 6. Fungsi Pelatihan dan Evaluasi

Bagian ini mengikuti ketentuan metrik per run di Spesifikasi dan aturan Slide 10 (validasi hanya untuk memilih epoch, uji dipakai sekali per run).

Alur setiap run: `set_seed(seed)`, membuat model dengan head baru, menerapkan strategi, lalu melatih sampai `EPOCHS` epoch. Di akhir setiap epoch model dievaluasi pada data validasi, dan bobot dengan F1 makro validasi tertinggi disimpan. Setelah pelatihan selesai, bobot terbaik dimuat kembali lalu dievaluasi satu kali pada data uji resmi.

Metrik yang dicatat otomatis: akurasi, F1 makro, F1 per kelas pada data uji, epoch terbaik dan F1 validasi per epoch, waktu latih (tanpa waktu validasi), waktu inferensi (ms/kalimat), jumlah parameter total dan yang dilatih, serta memori GPU puncak.

In [ ]:
def sinkron():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def ke_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def prediksi(model, loader):
    # Prediksi sekaligus mengukur waktu inferensi (ms/kalimat); ada satu batch pemanasan dulu
    model.eval()
    warm = ke_device({k: v for k, v in next(iter(loader)).items() if k != "labels"})
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        model(**warm)
    sinkron(); t0 = time.perf_counter()
    preds, n = [], 0
    for batch in loader:
        inp = ke_device({k: v for k, v in batch.items() if k != "labels"})
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**inp).logits
        preds.append(logits.argmax(-1).cpu()); n += logits.size(0)
    sinkron()
    return torch.cat(preds).numpy(), (time.perf_counter() - t0) * 1000 / n

def latih_dan_evaluasi(bk, st, seed):
    set_seed(seed)                                         # seed untuk python, numpy, torch, cuda
    model, _ = buat_model(BACKBONES[bk]["hf"])             # head replacement
    info_param = terapkan_strategi(model, st)              # freezing sesuai strategi
    model.to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)             # urutan shuffle mengikuti seed
    dl_train = DataLoader(DATA[bk]["train"], batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=COLLATORS[bk], generator=g)
    dl_valid = DataLoader(DATA[bk]["valid"], batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=COLLATORS[bk])
    dl_test  = DataLoader(DATA[bk]["test"],  batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=COLLATORS[bk])

    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    dilatih = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    grup = [{"params": [p for n, p in dilatih if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
            {"params": [p for n, p in dilatih if any(nd in n for nd in no_decay)], "weight_decay": 0.0}]
    lr = STRATEGIES[st]["lr"]
    optimizer = torch.optim.AdamW(grup, lr=lr)
    total_step = len(dl_train) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_step), total_step)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    waktu_latih, loss_epoch, f1_valid_epoch = 0.0, [], []
    f1_valid_terbaik, epoch_terbaik, state_terbaik = -1.0, 0, None
    for ep in range(EPOCHS):
        sinkron(); t0 = time.perf_counter()
        model.train(); total_loss, n = 0.0, 0
        for batch in dl_train:
            batch = ke_device(batch)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for _, p in dilatih], GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            total_loss += loss.item() * batch["labels"].size(0); n += batch["labels"].size(0)
        sinkron(); waktu_latih += time.perf_counter() - t0          # waktu validasi tidak ikut dihitung
        loss_epoch.append(total_loss / n)
        # Validasi hanya dipakai untuk memilih epoch terbaik (Slide 10)
        y_val_pred, _ = prediksi(model, dl_valid)
        f1_v = f1_score(y_valid, y_val_pred, average="macro", zero_division=0)
        f1_valid_epoch.append(f1_v)
        if f1_v > f1_valid_terbaik:
            f1_valid_terbaik, epoch_terbaik = f1_v, ep + 1
            state_terbaik = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
    mem_puncak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else float("nan")

    # Evaluasi di data uji sekali saja, memakai bobot epoch terbaik
    model.load_state_dict(state_terbaik)
    y_pred, infer_ms = prediksi(model, dl_test)
    f1_kelas = f1_score(y_test, y_pred, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
    hasil = {
        "kode": KODE, "dataset": DATASET, "mode": MODE, "run_id": f"{bk}|{st}|{seed}",
        "backbone": bk, "backbone_hf": BACKBONES[bk]["hf"], "strategi": st, "seed": seed,
        "lr": lr, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
        "n_train": len(DATA[bk]["train"]), "n_valid": len(DATA[bk]["valid"]), "n_test": len(DATA[bk]["test"]),
        "best_epoch": epoch_terbaik, "f1_valid_best": f1_valid_terbaik,
        "f1_valid_epoch": json.dumps([round(x, 4) for x in f1_valid_epoch]),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        **{f"f1_{lab}": f1_kelas[i] for i, lab in enumerate(LABELS)},
        "train_time_s": waktu_latih, "infer_ms_per_kalimat": infer_ms,
        **info_param, "peak_mem_mb": mem_puncak,
        "loss_epoch": json.dumps([round(x, 4) for x in loss_epoch]),
        "gpu": GPU_NAME, "selesai_pada": datetime.now().isoformat(timespec="seconds"),
    }
    del model, optimizer, scheduler, scaler, state_terbaik
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return hasil, y_pred


## 7. Matriks Eksperimen (Soal 2 dan Soal 3)

Sesuai C.2 dan C.3, ketiga backbone diuji dengan protokol yang sama persis, setiap kombinasi dijalankan dengan 3 seed, dan setiap backbone dicoba dengan semua strategi.

- Setiap run langsung ditambahkan ke `hasil_<dataset>_<kode>.csv`, dan prediksinya disimpan ke `prediksi_<dataset>_<kode>.json`.
- Run yang sudah tercatat di CSV dilewati, jadi aman dilanjutkan setelah sesi terputus.
- Urutan perulangan: backbone, lalu strategi, lalu seed.
- Skor yang dicetak (`acc`, `F1`) adalah skor pada data uji resmi dengan bobot dari epoch terbaik (`ep*`) yang dipilih lewat validasi.
- Tabel setelahnya merangkum hasil akhir matriks (kolom `f1_macro`) setelah semua run selesai.

In [ ]:
cek_rancangan(tegas=(MODE == "penuh"))   # mode penuh berhenti di sini kalau rancangan belum lengkap

def daftar_run():
    for bk in BACKBONES:
        if JALANKAN_BACKBONE and bk not in JALANKAN_BACKBONE:
            continue
        for st in STRATEGIES:
            if JALANKAN_STRATEGI and st not in JALANKAN_STRATEGI:
                continue
            for seed in SEEDS:
                yield bk, st, seed

def muat_hasil():
    return pd.read_csv(HASIL_CSV) if HASIL_CSV.exists() else pd.DataFrame()

def simpan_baris(baris):
    pd.DataFrame([baris]).to_csv(HASIL_CSV, mode="a", header=not HASIL_CSV.exists(), index=False)

PREDIKSI = json.loads(PRED_JSON.read_text()) if PRED_JSON.exists() else {}
selesai = set(muat_hasil().get("run_id", pd.Series(dtype=str)))
antrian = [r for r in daftar_run() if f"{r[0]}|{r[1]}|{r[2]}" not in selesai]
print(f"Total run di matriks: {len(list(daftar_run()))} | sudah selesai: {len(selesai)} | akan dijalankan: {len(antrian)}\n")

t_mulai = time.time()
for i, (bk, st, seed) in enumerate(antrian, 1):
    print(f"[{i}/{len(antrian)}] {bk:12s} | {st:8s} | seed {seed} ...", end=" ", flush=True)
    hasil, y_pred = latih_dan_evaluasi(bk, st, seed)
    simpan_baris(hasil)
    PREDIKSI[hasil["run_id"]] = y_pred.tolist()
    PRED_JSON.write_text(json.dumps(PREDIKSI))
    sisa = (time.time() - t_mulai) / i * (len(antrian) - i)
    print(f"ep*={hasil['best_epoch']} | uji acc={hasil['accuracy']:.4f} | F1={hasil['f1_macro']:.4f} | latih={hasil['train_time_s']:.0f}s "
          f"| infer={hasil['infer_ms_per_kalimat']:.2f} ms/kal | sisa ~{sisa/60:.1f} menit")

print(f"\nSelesai. Semua run tersimpan di {HASIL_CSV}")
hasil_df = muat_hasil()
hasil_df[["run_id", "best_epoch", "f1_valid_best", "accuracy", "f1_macro", "train_time_s", "infer_ms_per_kalimat",
          "params_trainable", "peak_mem_mb"]].round(4)

In [ ]:
# Ringkasan F1 makro per backbone dan strategi
try:
    ringkas_bk = (hasil_df.pivot_table(index=["backbone", "backbone_hf"], columns="strategi",
                                       values="f1_macro", aggfunc="mean").round(4))
    ringkas_bk.index = ringkas_bk.index.droplevel(1)
    print("Rata-rata F1 makro per kombinasi (baris = backbone, kolom = strategi):")
    display(ringkas_bk)
except Exception as e:
    print("Ringkasan belum bisa dibuat (hasil belum lengkap):", e)

TAMPIL = [c for c in ["run_id", "backbone", "strategi", "seed", "lr", "best_epoch", "f1_valid_best", "accuracy", "f1_macro",
                      "train_time_s", "infer_ms_per_kalimat", "params_total", "params_trainable",
                      "persen_trainable", "peak_mem_mb"] if c in hasil_df]
print(f"Total baris hasil: {len(hasil_df)}")
hasil_df[TAMPIL].round(5)

## 8. (Opsional) Menggabungkan Hasil Antar-Anggota

Bagian ini hanya untuk mendukung pembagian kerja kelompok. Karena seluruh matriks kami jalankan di notebook ini, bagian ini akan dilewati.

Jika matriks dibagi antar-anggota:
1. Setiap anggota mengunduh `hasil_smsa_<KODE>.csv` dan `prediksi_smsa_<KODE>.json` miliknya.
2. Unggah semua berkas tersebut ke folder `hasil_studio3/gabung/` di notebook utama. Nama berkas boleh diberi akhiran, misalnya `hasil_smsa_<KODE>_anggota2.csv`.
3. Jalankan sel di bawah. Sel ini akan menolak berkas dengan kode konfigurasi yang berbeda.

Waktu inferensi dari sesi yang berbeda tidak boleh dibandingkan langsung. Karena itu bagian 9 mengukur ulang waktu inferensi semua backbone dalam satu sesi.

In [ ]:
GABUNG_DIR = OUTPUT_DIR / "gabung"; GABUNG_DIR.mkdir(exist_ok=True)
csv_lain = sorted(GABUNG_DIR.glob(f"hasil_{DATASET}_*.csv"))
if not csv_lain:
    print("Tidak ada file di folder gabung/, bagian ini dilewati (wajar jika semua run dikerjakan di notebook ini).")
else:
    semua = [muat_hasil()] + [pd.read_csv(f) for f in csv_lain]
    gabungan = pd.concat([d for d in semua if len(d)], ignore_index=True)
    kode_lain = set(gabungan["kode"]) - {KODE}
    assert not kode_lain, f"Ada hasil dengan konfigurasi berbeda: {kode_lain}. Samakan konfigurasinya dulu."
    gabungan = gabungan.drop_duplicates(subset="run_id", keep="first")
    gabungan.to_csv(HASIL_CSV, index=False)
    for f in GABUNG_DIR.glob(f"prediksi_{DATASET}_*.json"):
        for k, v in json.loads(f.read_text()).items():
            PREDIKSI.setdefault(k, v)
    PRED_JSON.write_text(json.dumps(PREDIKSI))
    print(f"Digabung: {len(gabungan)} run unik dari {len(csv_lain) + 1} sumber, disimpan ke {HASIL_CSV.name}")

## 9. Benchmark Waktu Inferensi dalam Satu Sesi

Sesuai Spesifikasi, waktu inferensi (ms/kalimat) hanya dibandingkan jika diukur dalam sesi yang sama.

Kecepatan inferensi ditentukan oleh arsitektur backbone, bukan oleh strategi atau bobot hasil latih, karena ketiga strategi menghasilkan arsitektur yang sama. Jadi setiap backbone cukup diukur sekali dalam satu sesi, dengan dua skenario:
- Batch 1: latensi per kalimat, yang relevan untuk pemakaian real-time.
- Batch `EVAL_BATCH_SIZE`: throughput, yang relevan untuk memproses banyak data sekaligus.

Setiap pengukuran diulang 3 kali dan yang diambil adalah nilai mediannya.

In [ ]:
N_LATENSI = min(200, len(test_df))   # jumlah kalimat untuk pengukuran batch 1
ULANG = 3
baris = []
for bk, cfg in BACKBONES.items():
    m, _ = buat_model(cfg["hf"]); m.to(DEVICE)
    sub = torch.utils.data.Subset(DATA[bk]["test"], range(N_LATENSI))
    dl1 = DataLoader(sub, batch_size=1, collate_fn=COLLATORS[bk])
    dlb = DataLoader(DATA[bk]["test"], batch_size=EVAL_BATCH_SIZE, collate_fn=COLLATORS[bk])
    lat = [prediksi(m, dl1)[1] for _ in range(ULANG)]
    thr = [prediksi(m, dlb)[1] for _ in range(ULANG)]
    baris.append({"backbone": bk, "params_total (juta)": round(sum(p.numel() for p in m.parameters()) / 1e6, 1),
                  "latensi_batch1_ms_per_kalimat": float(np.median(lat)),
                  f"throughput_batch{EVAL_BATCH_SIZE}_ms_per_kalimat": float(np.median(thr)),
                  "gpu": GPU_NAME})
    del m; gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
INFER_DF = pd.DataFrame(baris)
INFER_DF.to_csv(INFER_CSV, index=False)
INFER_DF.round(3)

## 10. Rekap Agregat (rata-rata ± simpangan baku antar-seed)

Sesuai ketentuan pelaporan hasil di Spesifikasi, yang kami laporkan adalah rata-rata ± simpangan baku antar-seed (simpangan baku sampel, `ddof=1`), bukan nilai terbaik dari satu run. Rekap ini disimpan ke `rekap_agregat_<dataset>_<kode>.csv`.

In [ ]:
hasil_df = muat_hasil()
assert len(hasil_df), "Belum ada hasil run. Jalankan bagian 7 dulu."
ORDER_BK = [b for b in BACKBONES if b in set(hasil_df["backbone"])]
ORDER_ST = [s for s in STRATEGIES if s in set(hasil_df["strategi"])]

METRIK = ["accuracy", "f1_macro", "f1_negative", "f1_neutral", "f1_positive", "f1_valid_best", "best_epoch",
          "train_time_s", "infer_ms_per_kalimat", "peak_mem_mb"]
agg = hasil_df.groupby(["backbone", "strategi"]).agg(
    n_seed=("seed", "nunique"),
    **{f"{m}_mean": (m, "mean") for m in METRIK},
    **{f"{m}_std": (m, "std") for m in METRIK},
    params_total=("params_total", "first"),
    params_trainable=("params_trainable", "first"),
    persen_trainable=("persen_trainable", "first"),
).reset_index()
agg["backbone"] = pd.Categorical(agg["backbone"], ORDER_BK, ordered=True)
agg["strategi"] = pd.Categorical(agg["strategi"], ORDER_ST, ordered=True)
agg = agg.sort_values(["backbone", "strategi"]).reset_index(drop=True)
agg.insert(0, "kode", KODE)
agg.to_csv(REKAP_CSV, index=False)
print(f"Rekap disimpan: {REKAP_CSV}")

def pm(m, s, k=4):
    return f"{m:.{k}f} ± {0 if pd.isna(s) else s:.{k}f}"

TABEL_UTAMA = pd.DataFrame({
    "Backbone": agg["backbone"].map(lambda b: BACKBONES[b]["nama"]),
    "Strategi": agg["strategi"].map(lambda s: STRATEGIES[s]["nama"]),
    "n seed": agg["n_seed"],
    "Akurasi": [pm(m, s) for m, s in zip(agg["accuracy_mean"], agg["accuracy_std"])],
    "F1 makro (uji)": [pm(m, s) for m, s in zip(agg["f1_macro_mean"], agg["f1_macro_std"])],
    "Δ vs TF-IDF (poin)": ((agg["f1_macro_mean"] - F1_TFIDF) * 100).round(2),
    "Epoch terbaik (rata2)": agg["best_epoch_mean"].round(2),
    "Waktu latih (s)": [pm(m, s, 1) for m, s in zip(agg["train_time_s_mean"], agg["train_time_s_std"])],
    "Inferensi (ms/kal)": [pm(m, s, 3) for m, s in zip(agg["infer_ms_per_kalimat_mean"], agg["infer_ms_per_kalimat_std"])],
    "Param total (jt)": (agg["params_total"] / 1e6).round(1),
    "Param dilatih": agg["params_trainable"].map("{:,}".format),
    "% dilatih": agg["persen_trainable"].round(2),
    "Memori puncak (MB)": agg["peak_mem_mb_mean"].round(0),
})
if (agg["n_seed"] < 3).any() and MODE == "penuh":
    print("PERINGATAN: ada kombinasi dengan < 3 seed, lengkapi dulu sebelum dilaporkan.")
TABEL_UTAMA

In [ ]:
# F1 makro (mean ± std) per backbone dan strategi
pivot = agg.pivot(index="backbone", columns="strategi", values="f1_macro_mean").loc[ORDER_BK, ORDER_ST]
pivot_std = agg.pivot(index="backbone", columns="strategi", values="f1_macro_std").loc[ORDER_BK, ORDER_ST]
pivot_txt = pivot.round(4).astype(str) + " ± " + pivot_std.fillna(0).round(4).astype(str)
pivot_txt.index = [BACKBONES[b]["nama"] for b in pivot_txt.index]
print("F1 makro (mean ± std antar-seed)")
pivot_txt

## 11. Visualisasi

Grafik yang dibuat: (a) performa setiap kombinasi backbone dan strategi, (b) trade-off antara performa dan biaya komputasi, (c) sebaran skor antar-seed dan kurva loss, serta (d) confusion matrix untuk konfigurasi terbaik.

In [ ]:
WARNA = {"full": "#1f77b4", "partial": "#ff7f0e", "probing": "#2ca02c"}
nama_bk = [BACKBONES[b]["nama"] for b in ORDER_BK]
x = np.arange(len(ORDER_BK)); w = 0.8 / len(ORDER_ST)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for j, (metrik, judul) in enumerate([("f1_macro", "F1 makro"), ("accuracy", "Akurasi")]):
    ax = axes[j]
    for i, st in enumerate(ORDER_ST):
        d = agg[agg["strategi"] == st].set_index("backbone").loc[ORDER_BK]
        b = ax.bar(x + (i - (len(ORDER_ST) - 1) / 2) * w, d[f"{metrik}_mean"], w,
                   yerr=d[f"{metrik}_std"].fillna(0), capsize=4, label=STRATEGIES[st]["nama"],
                   color=WARNA.get(st))
        ax.bar_label(b, fmt="%.3f", fontsize=8, padding=2)
    ax.set_xticks(x, nama_bk); ax.set_ylabel(judul); ax.set_title(f"{judul} (mean ± std, {len(SEEDS)} seed)")
    acuan = {"f1_macro": (F1_TFIDF, F1_ACAK), "accuracy": tuple(BASELINE_DF.loc[[0, 2], "accuracy"])}[metrik]
    ax.axhline(acuan[0], color="crimson", ls="--", lw=1.2, label="TF-IDF + LogReg (latih penuh)")
    ax.axhline(acuan[1], color="gray", ls=":", lw=1.2, label="Tebakan acak")
    ax.set_ylim(max(0, min(agg[f"{metrik}_mean"].min(), acuan[1]) - 0.08), 1.0)
axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"grafik_performa_{KODE}.png", dpi=150); plt.show()

In [ ]:
# Trade-off: performa vs biaya komputasi
MARKER = {"full": "o", "partial": "s", "probing": "^"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
lat = INFER_DF.set_index("backbone")["latensi_batch1_ms_per_kalimat"] if "INFER_DF" in globals() else None
for _, r in agg.iterrows():
    lab = f"{BACKBONES[r['backbone']]['nama']}-{r['strategi']}"
    kw = dict(marker=MARKER.get(r["strategi"], "o"), s=60 + r["params_total"] / 2e6, alpha=.8)
    axes[0].errorbar(r["train_time_s_mean"], r["f1_macro_mean"], yerr=r["f1_macro_std"], fmt="none", ecolor="gray")
    axes[0].scatter(r["train_time_s_mean"], r["f1_macro_mean"], **kw); axes[0].annotate(lab, (r["train_time_s_mean"], r["f1_macro_mean"]), fontsize=7)
    xi = lat[r["backbone"]] if lat is not None else r["infer_ms_per_kalimat_mean"]
    axes[1].scatter(xi, r["f1_macro_mean"], **kw); axes[1].annotate(lab, (xi, r["f1_macro_mean"]), fontsize=7)
    axes[2].scatter(r["params_trainable"], r["f1_macro_mean"], **kw); axes[2].annotate(lab, (r["params_trainable"], r["f1_macro_mean"]), fontsize=7)
axes[0].set(xlabel="Waktu latih per run (s)", ylabel="F1 makro", title="F1 vs waktu latih")
axes[1].set(xlabel="Latensi inferensi batch-1 (ms/kalimat, satu sesi)", ylabel="F1 makro", title="F1 vs kecepatan inferensi")
axes[2].set(xlabel="Parameter dilatih (skala log)", ylabel="F1 makro", title="F1 vs parameter dilatih", xscale="log")
for ax in axes: ax.grid(alpha=.3)
plt.suptitle("Trade-off akurasi vs biaya (ukuran titik menunjukkan jumlah parameter)", y=1.02)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"grafik_tradeoff_{KODE}.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Variasi antar-seed + kurva loss latih
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
for i, st in enumerate(ORDER_ST):
    for k, bk in enumerate(ORDER_BK):
        v = hasil_df[(hasil_df["backbone"] == bk) & (hasil_df["strategi"] == st)]["f1_macro"]
        axes[0].scatter(np.full(len(v), k + (i - 1) * 0.22), v, color=WARNA.get(st),
                        label=STRATEGIES[st]["nama"] if k == 0 else None)
axes[0].set_xticks(range(len(ORDER_BK)), nama_bk); axes[0].set_ylabel("F1 makro")
axes[0].set_title("F1 makro per seed (sebaran antar-seed)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)

for _, r in hasil_df[hasil_df["seed"] == SEEDS[0]].iterrows():
    le = json.loads(r["loss_epoch"])
    axes[1].plot(range(1, len(le) + 1), le, marker="o", color=WARNA.get(r["strategi"]),
                 ls={0: "-", 1: "--", 2: ":", 3: "-."}[ORDER_BK.index(r["backbone"]) % 4],
                 label=f"{BACKBONES[r['backbone']]['nama']}-{r['strategi']}")
axes[1].set(xlabel="Epoch", ylabel="Loss latih", title=f"Kurva loss latih (seed {SEEDS[0]})")
axes[1].legend(fontsize=7, ncol=2); axes[1].grid(alpha=.3)
for _, r in hasil_df[hasil_df["seed"] == SEEDS[0]].iterrows():
    fv = json.loads(r["f1_valid_epoch"])
    axes[2].plot(range(1, len(fv) + 1), fv, marker="o", color=WARNA.get(r["strategi"]),
                 ls={0: "-", 1: "--", 2: ":", 3: "-."}[ORDER_BK.index(r["backbone"]) % 4])
axes[2].set(xlabel="Epoch", ylabel="F1 makro validasi", title=f"F1 validasi per epoch (dasar pemilihan epoch, seed {SEEDS[0]})")
axes[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"grafik_seed_loss_{KODE}.png", dpi=150); plt.show()

In [ ]:
# Confusion matrix konfigurasi terbaik, dijumlahkan dari semua seed
terbaik = agg.sort_values("f1_macro_mean", ascending=False).iloc[0]
bk_t, st_t = terbaik["backbone"], terbaik["strategi"]
cm = np.zeros((NUM_LABELS, NUM_LABELS), dtype=int)
for seed in SEEDS:
    rid = f"{bk_t}|{st_t}|{seed}"
    if rid in PREDIKSI:
        cm += confusion_matrix(y_test, PREDIKSI[rid], labels=list(range(NUM_LABELS)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center",
                color="white" if cm_norm[i, j] > .5 else "black", fontsize=9)
ax.set_xticks(range(NUM_LABELS), LABELS); ax.set_yticks(range(NUM_LABELS), LABELS)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Label sebenarnya")
ax.set_title(f"Terbaik: {BACKBONES[bk_t]['nama']} + {STRATEGIES[st_t]['nama']}\n(uji resmi, gabungan {len(SEEDS)} seed)", fontsize=10)
plt.colorbar(im, fraction=.046); plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"grafik_confusion_{KODE}.png", dpi=150); plt.show()

rid0 = f"{bk_t}|{st_t}|{SEEDS[0]}"
if rid0 in PREDIKSI:
    print(f"Classification report ({rid0}):\n")
    print(classification_report(y_test, PREDIKSI[rid0], target_names=LABELS, digits=4))

## 12. Pengujian Hipotesis H1–H3

Hipotesis yang sudah ditulis di Rancangan (bagian 3) diuji di sini menggunakan hasil eksperimen.

Setiap hipotesis dicek dengan dua cara:
- Arah selisih, misalnya apakah IndoBERT memang lebih tinggi daripada mBERT. Bagian ini dianggap lolos jika arahnya sesuai dan selisihnya lebih besar daripada jumlah simpangan baku kedua kelompok (SD₁ + SD₂). Jika arahnya sesuai tetapi selisihnya lebih kecil dari itu, hasilnya ditandai "dalam variasi seed".
- Besar selisih, untuk hipotesis yang menyebut batas angka. Pada H2 penurunan F1 paling banyak 3 poin, dan pada H3 partial paling jauh 2 poin di bawah full.

Sebuah hipotesis dinyatakan didukung jika semua bagiannya lolos, didukung sebagian jika hanya sebagian yang lolos, dan tidak didukung jika tidak ada yang lolos. Hasil Welch t-test ditampilkan sebagai pelengkap. Karena tiap kelompok hanya berisi 3 seed, nilai p-nya hanya bisa dibaca sebagai petunjuk, bukan bukti kuat.

In [ ]:
try:
    from scipy.stats import ttest_ind
except ImportError:
    ttest_ind = None

# Fungsi bantu
def nilai(bk, st, m="f1_macro"):
    return hasil_df[(hasil_df["backbone"] == bk) & (hasil_df["strategi"] == st)][m].to_numpy()

def rata(bk, st, m="f1_macro"):
    v = nilai(bk, st, m)
    return v.mean() if len(v) else float("nan")

def sd_seed(v):
    return v.std(ddof=1) if len(v) > 1 else 0.0

def uji_arah(a, b, nama_a, nama_b, arah="lebih_besar", m="f1_macro"):
    # Cek arah selisih a - b: lolos bila arahnya sesuai dan |selisih| > SD_a + SD_b
    d, batas = a.mean() - b.mean(), sd_seed(a) + sd_seed(b)
    sesuai = d > 0 if arah == "lebih_besar" else d < 0
    p = f" | Welch p = {ttest_ind(a, b, equal_var=False).pvalue:.3f}" if ttest_ind and len(a) > 1 and len(b) > 1 else ""
    status = "lolos" if sesuai and abs(d) > batas else ("dalam variasi seed" if sesuai else "tidak lolos")
    print(f"   {m}: {nama_a} {a.mean():.4f}±{sd_seed(a):.4f} vs {nama_b} {b.mean():.4f}±{sd_seed(b):.4f} "
          f", selisih {d:+.4f}{p}  [{status}]")
    return status == "lolos"

def uji_batas(selisih_poin, batas_poin, keterangan):
    # Cek besar selisih terhadap batas angka yang ditulis di hipotesis
    ok = selisih_poin <= batas_poin
    print(f"   {keterangan}: {selisih_poin:+.2f} poin (batas <= {batas_poin})  [{'sesuai batas' if ok else 'melewati batas'}]")
    return ok

def vonis(hasil):
    return "DIDUKUNG" if all(hasil) else ("DIDUKUNG SEBAGIAN" if any(hasil) else "tidak DIDUKUNG")

# H1
print("H1: IndoBERT > mBERT (F1 makro, full FT)")
h1 = [uji_arah(nilai("indobert", "full"), nilai("mbert", "full"), "IndoBERT", "mBERT")]
if PAKAI_XLMR and "xlmr" in ORDER_BK:
    uji_arah(nilai("indobert", "full"), nilai("xlmr", "full"), "IndoBERT", "XLM-R")   # pembanding tambahan
print(f"   Kesimpulan H1: {vonis(h1)}")

# H2
print("\nH2: DistilmBERT lebih cepat dari mBERT, penurunan F1 <= 3 poin (full FT)")
h2 = [uji_arah(nilai("distilmbert", "full", "train_time_s"), nilai("mbert", "full", "train_time_s"),
               "DistilmBERT", "mBERT", arah="lebih_kecil", m="train_time_s")]
if "INFER_DF" in globals():
    L = INFER_DF.set_index("backbone")["latensi_batch1_ms_per_kalimat"]
    h2.append(L["distilmbert"] < L["mbert"])
    print(f"   latensi batch-1 (satu sesi): DistilmBERT {L['distilmbert']:.2f} ms vs mBERT {L['mbert']:.2f} ms "
          f"({(1 - L['distilmbert'] / L['mbert']) * 100:.1f}% lebih cepat)  [{'lolos' if h2[-1] else 'tidak lolos'}]")
h2.append(uji_batas((rata("mbert", "full") - rata("distilmbert", "full")) * 100, 3, "penurunan F1 makro"))
print(f"   Kesimpulan H2: {vonis(h2)}")

# H3
print("\nH3: full >= partial > probing; partial mendekati full (<= 2 poin)")
vonis_bk = []
for bk in ORDER_BK:
    urut = " > ".join(sorted(ORDER_ST, key=lambda s: -rata(bk, s)))
    print(f" - {BACKBONES[bk]['nama']} (urutan aktual: {urut})")
    hasil = []
    if {"full", "partial"} <= set(ORDER_ST):
        f, pa = nilai(bk, "full"), nilai(bk, "partial")
        d = (f.mean() - pa.mean()) * 100
        # full >= partial dianggap terpenuhi selama partial tidak lebih tinggi secara bermakna
        hasil.append(d >= -(sd_seed(f) + sd_seed(pa)) * 100)
        print(f"   full >= partial: selisih {d:+.2f} poin  [{'lolos' if hasil[-1] else 'tidak lolos, partial lebih tinggi secara bermakna'}]")
        hasil.append(uji_batas(d, 2, "full - partial"))
    if {"partial", "probing"} <= set(ORDER_ST):
        hasil.append(uji_arah(nilai(bk, "partial"), nilai(bk, "probing"), "partial", "probing"))
    vonis_bk.append(vonis(hasil))
    print(f"   Kesimpulan {BACKBONES[bk]['nama']}: {vonis_bk[-1]}")
print(f"\nKesimpulan H3 (semua backbone): {vonis([v == 'DIDUKUNG' for v in vonis_bk])}")

### 12.1 Tabel bukti untuk 3 Pertanyaan Riset Wajib (Q1–Q3)

Kriteria yang dipakai sama dengan yang sudah ditetapkan di Rancangan (bagian 3.6) sebelum eksperimen:
- Q1: F1 makro ± SD, waktu latih, waktu inferensi (latensi batch-1 dan throughput batch-64 dalam satu sesi), dan jumlah parameter. Kolom Pareto bernilai "ya" jika tidak ada konfigurasi lain yang sekaligus lebih akurat dan lebih cepat inferensinya.
- Q2: selisih dengan full fine-tuning dianggap bermakna jika |Δ| > SD₁ + SD₂. Strategi disebut kompetitif jika selisihnya tidak bermakna atau paling banyak 2 poin, dan skornya tetap di atas TF-IDF.
- Q3: syarat minimum F1 di atas TF-IDF, lalu skor = 0,5 × F1 + 0,25 × kecepatan inferensi + 0,25 × hemat waktu latih (dinormalisasi min–max).

In [ ]:
# Q1: trade-off akurasi dan efisiensi per backbone
LAT = INFER_DF.set_index("backbone")
thr_col = [c for c in INFER_DF.columns if c.startswith("throughput")][0]
q1 = agg[agg["strategi"] == "full"].copy()
Q1_TABEL = pd.DataFrame({
    "Backbone": q1["backbone"].map(lambda b: BACKBONES[b]["nama"]),
    "F1 makro uji (full FT)": [pm(m, s) for m, s in zip(q1["f1_macro_mean"], q1["f1_macro_std"])],
    "Akurasi uji": [pm(m, s) for m, s in zip(q1["accuracy_mean"], q1["accuracy_std"])],
    "Waktu latih (s)": [pm(m, s, 1) for m, s in zip(q1["train_time_s_mean"], q1["train_time_s_std"])],
    "Latensi batch-1 (ms/kal)": q1["backbone"].map(LAT["latensi_batch1_ms_per_kalimat"]).round(2).values,
    "Throughput batch-64 (ms/kal)": q1["backbone"].map(LAT[thr_col]).round(3).values,
    "Parameter total (jt)": (q1["params_total"] / 1e6).round(1).values,
    "Memori puncak (MB)": q1["peak_mem_mb_mean"].round(0).values,
})
print("Q1: Backbone pada full fine-tuning (protokol identik, uji resmi 500 kalimat)")
display(Q1_TABEL)

# Batas Pareto dari semua konfigurasi: F1 tinggi dan latensi batch-1 rendah
pts = agg.assign(lat=agg["backbone"].map(LAT["latensi_batch1_ms_per_kalimat"]).astype(float))
def didominasi(r):
    return any((o.f1_macro_mean >= r.f1_macro_mean) and (o.lat <= r.lat) and
               ((o.f1_macro_mean > r.f1_macro_mean) or (o.lat < r.lat)) for o in pts.itertuples())
pts["Pareto (F1 vs latensi)"] = ["ya" if not didominasi(r) else "" for r in pts.itertuples()]
print("Konfigurasi pada batas Pareto F1-vs-latensi:")
display(pts.loc[pts["Pareto (F1 vs latensi)"] == "ya", ["backbone", "strategi", "f1_macro_mean", "f1_macro_std", "lat", "train_time_s_mean"]].round(4))

In [ ]:
# Q2: apakah partial/probing kompetitif dibanding full FT? Selisih dibandingkan dengan simpangan antar-seed
baris = []
for bk in ORDER_BK:
    f = nilai(bk, "full")
    for st in [s for s in ORDER_ST if s != "full"]:
        v = nilai(bk, st)
        d = v.mean() - f.mean(); sd = f.std(ddof=1) + v.std(ddof=1)
        p = ttest_ind(v, f, equal_var=False).pvalue if ttest_ind else float("nan")
        bermakna = abs(d) > sd
        kompetitif = ((not bermakna) or abs(d) <= 0.02) and v.mean() > F1_TFIDF
        hemat = 1 - rata(bk, st, "train_time_s") / rata(bk, "full", "train_time_s")
        baris.append({"Backbone": BACKBONES[bk]["nama"], "Strategi": st,
                      "F1 strategi": pm(v.mean(), v.std(ddof=1)), "F1 full": pm(f.mean(), f.std(ddof=1)),
                      "Δ F1 (poin)": round(d * 100, 2), "SD₁+SD₂ (poin)": round(sd * 100, 2),
                      "|Δ| / (SD₁+SD₂)": round(abs(d) / sd, 1) if sd > 0 else float("inf"),
                      "Welch p": round(p, 4),
                      "Selisih bermakna?": "ya" if bermakna else "belum beda",
                      "> TF-IDF?": "ya" if v.mean() > F1_TFIDF else "tidak",
                      "Hemat waktu latih": f"{hemat*100:.1f}%",
                      "Kompetitif?": "ya" if kompetitif else "tidak"})
Q2_TABEL = pd.DataFrame(baris)
print(f"Q2: Strategi hemat vs full FT (acuan TF-IDF latih penuh = {F1_TFIDF:.4f})")
Q2_TABEL

In [ ]:
# Q3: skor gabungan sesuai kriteria di Rancangan
def minmax(x, besar_lebih_baik=True):
    x = np.asarray(x, dtype=float); r = (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.ones_like(x)
    return r if besar_lebih_baik else 1 - r

q3 = pts.copy()
q3["lolos_syarat (F1 > TF-IDF)"] = q3["f1_macro_mean"] > F1_TFIDF
q3["s_F1"] = minmax(q3["f1_macro_mean"])
q3["s_inferensi"] = minmax(q3["lat"], besar_lebih_baik=False)
q3["s_latih"] = minmax(q3["train_time_s_mean"], besar_lebih_baik=False)
q3["skor"] = 0.5 * q3["s_F1"] + 0.25 * q3["s_inferensi"] + 0.25 * q3["s_latih"]
q3.loc[~q3["lolos_syarat (F1 > TF-IDF)"], "skor"] = np.nan          # tidak memenuhi syarat minimum
Q3_TABEL = (q3.assign(Backbone=q3["backbone"].map(lambda b: BACKBONES[b]["nama"]),
                      F1=[pm(m, s) for m, s in zip(q3["f1_macro_mean"], q3["f1_macro_std"])])
              .sort_values("skor", ascending=False, na_position="last")
              [["Backbone", "strategi", "F1", "lat", "train_time_s_mean", "params_trainable",
                "lolos_syarat (F1 > TF-IDF)", "s_F1", "s_inferensi", "s_latih", "skor"]]
              .rename(columns={"lat": "latensi b1 (ms)", "train_time_s_mean": "waktu latih (s)"})
              .round(3).reset_index(drop=True))
Q3_TABEL.index += 1
print("Q3: Peringkat kombinasi (bobot: 0,5 F1 + 0,25 inferensi + 0,25 waktu latih; NaN = gugur syarat F1 > TF-IDF)")
display(Q3_TABEL)
# Uji sensitivitas bobot: apakah juara berubah jika bobot akurasi diubah?
# Sama seperti tabel di atas, hanya konfigurasi yang lolos syarat F1 > TF-IDF yang dihitung.
print("Uji sensitivitas bobot (hanya yang lolos syarat F1 > TF-IDF):")
lolos = q3["lolos_syarat (F1 > TF-IDF)"]
nama = lambda r: f"{BACKBONES[r['backbone']]['nama']} + {r['strategi']}"
for wf in [0.33, 0.5, 0.7]:
    sk = (wf * q3["s_F1"] + (1 - wf) / 2 * (q3["s_inferensi"] + q3["s_latih"]))[lolos].nlargest(2)
    j1, j2 = q3.loc[sk.index[0]], q3.loc[sk.index[1]]
    print(f"   bobot F1 = {wf:.2f} : juara {nama(j1)} ({sk.iloc[0]:.3f}) | "
          f"runner-up: {nama(j2)} ({sk.iloc[1]:.3f}) | selisih {sk.iloc[0] - sk.iloc[1]:.3f}")

## 13. Analisis, Rekomendasi, dan Kesimpulan

*(diisi setelah semua run selesai)*

## 14. Kepatuhan Spesifikasi dan Berkas yang Dikumpulkan

Sel berikut memeriksa secara otomatis apakah kriteria selesai untuk Soal 1, 2, dan 3 serta ketentuan berkas yang dikumpulkan sudah terpenuhi, terutama bagian yang baru bisa diperiksa setelah ada hasil run. Setelah itu semua berkas hasil dikemas menjadi satu arsip.

In [ ]:
# Pemeriksaan kepatuhan spesifikasi berdasarkan hasil run
hasil_df = muat_hasil()
print(f"Pemeriksaan kepatuhan spesifikasi (mode {MODE}, {'berkas uji jalan' if MODE == 'uji' else 'berkas resmi'}):")
if hasil_df.empty:
    print("PERINGATAN: belum ada hasil run. Jalankan bagian 7 dulu, lalu jalankan ulang sel ini.")
else:
    per_sel = hasil_df.groupby(["backbone", "strategi"])["seed"].nunique()
    cek = {
        "Soal 1: rancangan lengkap & dikonfirmasi": RANCANGAN_DIKONFIRMASI and all(str(v).strip() for v in RANCANGAN.values()),
        "Soal 2: >= 3 backbone diuji": hasil_df["backbone"].nunique() >= 3,
        "Soal 2: BERT, DistilBERT, IndoBERT ada": {"mbert", "distilmbert", "indobert"} <= set(hasil_df["backbone"]),
        "Soal 2: tiap kombinasi >= 3 seed": bool(len(per_sel)) and per_sel.min() >= 3,
        "Soal 2: tabel memuat akurasi, F1, waktu latih, inferensi, parameter": all(
            c in hasil_df for c in ["accuracy", "f1_macro", "train_time_s", "infer_ms_per_kalimat", "params_total"]),
        "Soal 3: full fine-tuning dijalankan": "full" in set(hasil_df["strategi"]),
        "Soal 3: probing/partial dijalankan": bool({"probing", "partial"} & set(hasil_df["strategi"])),
        "Soal 3: >=1 backbone dengan semua strategi": any(
            set(hasil_df[hasil_df["backbone"] == b]["strategi"]) >= set(STRATEGIES) for b in BACKBONES),
        "Soal 3: LR berbeda per strategi": hasil_df.groupby("strategi")["lr"].first().nunique() == hasil_df["strategi"].nunique(),
        "Hal teknis: berkas hasil & rekap dibuat otomatis": HASIL_CSV.exists() and REKAP_CSV.exists(),
        "Hal teknis: inferensi diukur dalam satu sesi": INFER_CSV.exists(),
        "Mode penuh (bukan uji jalan)": MODE == "penuh",
        "Slide 10: data latih 3.000-6.000": hasil_df["n_train"].between(3000, 6000).all(),
        "Slide 10: validasi >= 500 (hanya untuk memilih epoch)": (hasil_df["n_valid"] >= 500).all(),
        "Slide 10: uji = seluruh 500 uji resmi": (hasil_df["n_test"] == 500).all(),
        "Slide 10: tidak ada tumpang-tindih latih/validasi/uji": sum(OVERLAP.values()) == 0,
        "Slide 11: tidak ada F1 > 0,97 (indikasi kebocoran)": hasil_df["f1_macro"].max() <= 0.97,
        "Slide 11: semua run jauh di atas tebakan acak": hasil_df["f1_macro"].min() > F1_ACAK + 0.1,
        "Slide 11: full FT setiap backbone di atas baseline TF-IDF": all(
            rata(b, "full", "f1_macro") > F1_TFIDF for b in ORDER_BK),
        "Slide 9: tabel bukti Q1-Q3 tersedia": all(k in globals() for k in ["Q1_TABEL", "Q2_TABEL", "Q3_TABEL"]),
    }
    for k, v in cek.items():
        print(("[OK]   " if v else "[GAGAL]"), k)
    di_bawah = [f"{b}|{s}" for b in ORDER_BK for s in ORDER_ST if rata(b, s, "f1_macro") <= F1_TFIDF]
    if di_bawah:
        print(f"Catatan: konfigurasi di bawah TF-IDF ({F1_TFIDF:.4f}): {di_bawah}. Slide 11: wajar untuk probing; LR probing sudah di batas atas rentang (5e-3).")
    if MODE == "uji":
        print("Catatan: ini berkas uji jalan (UJI_JALAN=True). Untuk laporan resmi: set UJI_JALAN=False, RANCANGAN_DIKONFIRMASI=True, lalu Run all.")

import zipfile
BERKAS_KODE = sorted(f for f in OUTPUT_DIR.glob(f"*{KODE}{SUFIKS}*") if f.is_file() and f.suffix != ".zip")
arsip = OUTPUT_DIR.parent / f"studio3_{DATASET}_{KODE}{SUFIKS}.zip"
with zipfile.ZipFile(arsip, "w", zipfile.ZIP_DEFLATED) as z:      # hanya berkas dari konfigurasi ini
    for f in BERKAS_KODE:
        z.write(f, f.name)
print(f"\nSemua berkas dikemas: {arsip}")
print("Isi arsip:")
for f in BERKAS_KODE:
    print(f"   - {f.name}")

# Daftar berkas yang dikumpulkan (Spesifikasi: Yang Perlu Dikumpulkan)
print("\nYang dikumpulkan ke portal e-learning:")
print("1. Link notebook Colab/Kaggle (akses: Anyone with the link can view).")
print("2. PDF notebook yang sudah dijalankan penuh (berisi kode, output, dan grafik).")
print("   Nama berkas: STUDIO3_Kelompok15_MuhammadFadhilahAkbar_MuhammadFajriFirdaus.pdf")
print(f"3. Berkas hasil: {HASIL_CSV.name} dan {REKAP_CSV.name} (+ {BASELINE_CSV.name}, arsip zip bila diperlukan).")
print("4. Slide (maksimal 15) + laporan singkat 3-4 halaman: rancangan, hasil, analisis, jawaban 3 Pertanyaan Riset Wajib.")

In [ ]:
# Simpan hasil ke Google Drive (khusus Colab). Biasanya OUTPUT_DIR sudah di Drive sejak bagian 2,
# jadi sel ini hanya menyalin kalau Drive gagal di-mount di awal.
import shutil
from pathlib import Path

if DI_DRIVE:
    print(f"Hasil sudah tersimpan langsung di Google Drive: {OUTPUT_DIR}")
    for f in BERKAS_KODE:
        print("   -", f.name)
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/Hasil_Studio3")
        tujuan = DRIVE_DIR / OUTPUT_DIR.name
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        if tujuan.exists() and tujuan.resolve() != OUTPUT_DIR.resolve():
            shutil.rmtree(tujuan)                      # ganti salinan lama
        shutil.copytree(OUTPUT_DIR, tujuan)
        print(f"Hasil disalin ke Google Drive: {tujuan}")
    except ImportError:
        print("Bukan di Colab: langkah Google Drive dilewati (di Kaggle pakai panel Output).")
    except Exception as e:
        print("PERINGATAN: gagal menyimpan ke Google Drive:", e)


In [ ]:
# Unduh hasil (Colab). Di Kaggle, ambil dari panel Output.
print(f"Arsip hasil: {arsip}")
try:
    from google.colab import files                     # hanya ada di Colab
    for f in [HASIL_CSV, REKAP_CSV, Path(arsip)]:
        files.download(str(f))
    print("\nUnduhan dikirim: hasil CSV, rekap agregat, dan arsip zip.")
except ImportError:
    print(f"\nBukan di Colab: berkas ada di {OUTPUT_DIR.resolve()} (Kaggle: panel Output).")